# Практична робота 8
## Групування і зведені таблиці: groupby, agg, pivot_table, crosstab

Варіант 7 — Івано-Франківськ

In [17]:
import numpy as np
import pandas as pd

In [18]:
np.random.seed(42)

base_temp = 7.8
amplitude = 12
city = "Ivano-Frankivsk"

rows = []

for year in [2021, 2022, 2023, 2024]:
    for month in range(1, 13):
        seasonal = amplitude * np.cos((month - 7) / 12 * 2 * np.pi)
        noise = np.random.normal(0, 1.0)

        rows.append({
            "місто": city,
            "рік": year,
            "місяць": month,
            "температура": round(base_temp + seasonal + noise, 1),
        })

climate = pd.DataFrame(rows)

climate

,місто,рік,місяць,температура
0,Ivano-Frankivsk,2021,1,-3.7
1,Ivano-Frankivsk,2021,2,-2.7
2,Ivano-Frankivsk,2021,3,2.4
3,Ivano-Frankivsk,2021,4,9.3
4,Ivano-Frankivsk,2021,5,13.6
5,Ivano-Frankivsk,2021,6,18.0
6,Ivano-Frankivsk,2021,7,21.4
7,Ivano-Frankivsk,2021,8,19.0
8,Ivano-Frankivsk,2021,9,13.3
9,Ivano-Frankivsk,2021,10,8.3


In [19]:
print("Кількість рядків:", len(climate))
print("Кількість стовпців:", len(climate.columns))
print("\nНазви стовпців:")
print(climate.columns.tolist())

Кількість рядків: 48
Кількість стовпців: 4

Назви стовпців:
['місто', 'рік', 'місяць', 'температура']


## Завдання 1. Групування за роками

За допомогою groupby() та agg() обчислюємо середню, мінімальну та максимальну температуру для кожного року.

In [20]:
year_stats = climate.groupby("рік")["температура"].agg(
    ["mean", "min", "max"]
)

year_stats

,mean,min,max
рік,,,
2021,8.091667,-3.7,21.4
2022,7.216667,-4.5,18.9
2023,7.600000,-4.7,20.0
2024,7.466667,-4.6,19.7


За результатами групування середня температура у 2021 році становить приблизно 8.09 °C, у 2022 році — 7.22 °C, у 2023 році — 7.60 °C, а у 2024 році — 7.47 °C.

Чіткого тренду на потепління протягом 2021–2024 років не спостерігається. Найвища середня температура була у 2021 році, після чого вона знизилася. У 2023–2024 роках значення дещо коливалися. Отже, у межах цього синтетичного набору даних зміни температури виглядають переважно як випадкові коливання, а не як стабільне потепління.

## Завдання 2. Групування за місяцями

Для кожного місяця визначаємо середню температуру та стандартне відхилення за чотири роки.

In [21]:
month_stats = climate.groupby("місяць")["температура"].agg(
    ["mean", "std"]
)

month_stats

,mean,std
місяць,,
1,-4.100,0.424264
2,-3.575,1.129528
3,0.900,1.023067
4,8.175,0.865544
5,13.525,0.727438
6,18.200,0.294392
7,19.800,1.116542
8,18.425,1.381726
9,13.675,1.250000


In [22]:
month_stats["std"].idxmax(), month_stats["std"].max()

(np.int64(8), np.float64(1.381725973797506))

Найбільше стандартне відхилення спостерігається у серпні — приблизно 1.38 °C. Це означає, що температури серпня найбільше відрізняються між чотирма роками.

Можна припустити, що така нестабільність пов'язана зі змінами погодних умов у кінці літа та переходом до осіннього періоду. Водночас у цьому наборі даних значення є синтетичними, тому конкретний максимум стандартного відхилення також залежить від випадкового шуму, який був згенерований за допомогою np.random.normal().

## Завдання 3. Зведена таблиця pivot_table()

Побудуємо таблицю, у якій місяці будуть індексом, роки — стовпцями, а значеннями — середні температури.

In [23]:
climate_pivot = climate.pivot_table(
    index="місяць",
    columns="рік",
    values="температура",
    aggfunc="mean"
)

climate_pivot

рік,2021,2022,2023,2024
місяць,,,,
1,-3.7,-4.0,-4.7,-4.0
2,-2.7,-4.5,-2.5,-4.6
3,2.4,0.1,0.6,0.5
4,9.3,7.2,8.2,8.0
5,13.6,12.8,13.2,14.5
6,18.0,18.5,17.9,18.4
7,21.4,18.9,19.2,19.7
8,19.0,16.8,20.0,17.9
9,13.3,15.3,13.8,12.3


Вихідний набір climate представлений у довгому (tidy) форматі: кожен рядок відповідає одному спостереженню за конкретний місяць і рік. Такий формат зручний для групування за допомогою groupby(), фільтрації та побудови графіків.

Зведена таблиця pivot_table() зручніша для читання людиною, оскільки дозволяє швидко порівняти температуру одного місяця між різними роками. Для подальшого групування та обробки даних зручнішим є довгий tidy-формат, оскільки рік і місяць залишаються окремими змінними.

In [24]:
def get_season(month):
    if month in [12, 1, 2]:
        return "зима"
    elif month in [3, 4, 5]:
        return "весна"
    elif month in [6, 7, 8]:
        return "літо"
    else:
        return "осінь"


climate["сезон"] = climate["місяць"].apply(get_season)

climate.head(15)

,місто,рік,місяць,температура,сезон
0,Ivano-Frankivsk,2021,1,-3.7,зима
1,Ivano-Frankivsk,2021,2,-2.7,зима
2,Ivano-Frankivsk,2021,3,2.4,весна
3,Ivano-Frankivsk,2021,4,9.3,весна
4,Ivano-Frankivsk,2021,5,13.6,весна
5,Ivano-Frankivsk,2021,6,18.0,літо
6,Ivano-Frankivsk,2021,7,21.4,літо
7,Ivano-Frankivsk,2021,8,19.0,літо
8,Ivano-Frankivsk,2021,9,13.3,осінь
9,Ivano-Frankivsk,2021,10,8.3,осінь


In [25]:
climate["тепліше_за_середнє"] = climate["температура"] > base_temp

climate.head(15)

,місто,рік,місяць,температура,сезон,тепліше_за_середнє
0,Ivano-Frankivsk,2021,1,-3.7,зима,False
1,Ivano-Frankivsk,2021,2,-2.7,зима,False
2,Ivano-Frankivsk,2021,3,2.4,весна,False
3,Ivano-Frankivsk,2021,4,9.3,весна,True
4,Ivano-Frankivsk,2021,5,13.6,весна,True
5,Ivano-Frankivsk,2021,6,18.0,літо,True
6,Ivano-Frankivsk,2021,7,21.4,літо,True
7,Ivano-Frankivsk,2021,8,19.0,літо,True
8,Ivano-Frankivsk,2021,9,13.3,осінь,True
9,Ivano-Frankivsk,2021,10,8.3,осінь,True


In [26]:
climate[["місяць", "температура", "сезон", "тепліше_за_середнє"]].head(20)

,місяць,температура,сезон,тепліше_за_середнє
0,1,-3.7,зима,False
1,2,-2.7,зима,False
2,3,2.4,весна,False
3,4,9.3,весна,True
4,5,13.6,весна,True
5,6,18.0,літо,True
6,7,21.4,літо,True
7,8,19.0,літо,True
8,9,13.3,осінь,True
9,10,8.3,осінь,True


In [27]:
season_temp_table = pd.crosstab(
    climate["сезон"],
    climate["тепліше_за_середнє"]
)

season_temp_table



тепліше_за_середнє,False,True
сезон,,
весна,5,7
зима,12,0
літо,0,12
осінь,7,5


За допомогою crosstab() було отримано таблицю частот, яка показує кількість спостережень для кожного сезону залежно від того, чи була температура вищою за середньорічну температуру 7.8 °C.

Результат загалом відповідає очікуванням: у літні місяці більшість температур є вищими за середньорічну, а взимку — нижчими. Весна та осінь мають змішаний розподіл, оскільки це перехідні сезони. Не всі літні значення обов'язково мають бути вищими за 7.8 °C, оскільки до синтетичних даних додано випадковий шум.

In [28]:
climate_pivot_direct = climate.pivot(
    index="місяць",
    columns="рік",
    values="температура"
)

climate_pivot_direct

рік,2021,2022,2023,2024
місяць,,,,
1,-3.7,-4.0,-4.7,-4.0
2,-2.7,-4.5,-2.5,-4.6
3,2.4,0.1,0.6,0.5
4,9.3,7.2,8.2,8.0
5,13.6,12.8,13.2,14.5
6,18.0,18.5,17.9,18.4
7,21.4,18.9,19.2,19.7
8,19.0,16.8,20.0,17.9
9,13.3,15.3,13.8,12.3


In [29]:
climate.groupby(["місяць", "рік"]).size().value_counts()

1    48
Name: count, dtype: int64

Метод pivot() успішно працює на наборі climate, оскільки для кожної комбінації "місяць" та "рік" існує рівно один рядок. Усього є 4 роки та 12 місяців, тому отримуємо 4 × 12 = 48 унікальних комбінацій.

У прикладі з кав'ярнями для однієї комбінації індексу та стовпця існувало декілька значень, тому pivot() не міг визначити, яке значення записати в клітинку та завершувався помилкою.

Щоб pivot() почав завершуватися помилкою і для кліматичних даних, потрібно додати ще один вимірювальний об'єкт, наприклад декілька метеостанцій для одного міста. Тоді для комбінації "місяць + рік" існувало б декілька температурних значень — по одному від кожної станції. У такому випадку pivot() отримав би дублікати.

Для таких даних можна використовувати pivot_table(), оскільки він дозволяє виконати агрегацію, наприклад обчислити середню температуру за допомогою aggfunc="mean".

In [30]:
climate_stations = pd.concat([
    climate.assign(станція="Station_A"),
    climate.assign(станція="Station_B")
], ignore_index=True)

climate_stations.head()

,місто,рік,місяць,температура,сезон,тепліше_за_середнє,станція
0,Ivano-Frankivsk,2021,1,-3.7,зима,False,Station_A
1,Ivano-Frankivsk,2021,2,-2.7,зима,False,Station_A
2,Ivano-Frankivsk,2021,3,2.4,весна,False,Station_A
3,Ivano-Frankivsk,2021,4,9.3,весна,True,Station_A
4,Ivano-Frankivsk,2021,5,13.6,весна,True,Station_A


In [31]:
climate_stations.groupby(["місяць", "рік"]).size().value_counts()

2    48
Name: count, dtype: int64

In [32]:
climate_stations.pivot(
    index="місяць",
    columns="рік",
    values="температура"
)

ValueError: Index contains duplicate entries, cannot reshape